# Exploratory Data Analysis - UNSW-NB15

This notebook is for **exploring** the data. The project does not depend on it:
training, evaluation and the dashboard all run from the `src/` scripts.

It reuses the same loading and validation code as the rest of the project, so
what you see here is exactly what the models see.

Run it from the project root (VS Code: open the file and pick the `.venv` kernel).

In [ ]:
import sys, pathlib
# Make `import src` work when the notebook runs from the notebooks/ folder
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src import config
from src.data_loader import load_labeled_dataset, data_quality_summary, train_test_overlap

## 1. Load and validate both official files

In [ ]:
train = load_labeled_dataset(config.TRAIN_FILE)
test = load_labeled_dataset(config.TEST_FILE)
train.shape, test.shape

In [ ]:
train.head()

## 2. Data quality: missing values, duplicates, class balance

In [ ]:
q = data_quality_summary(train)
{k: q[k] for k in ["rows", "missing_values_total", "duplicate_rows_features_and_label",
                   "conflicting_duplicate_groups", "class_counts", "attack_fraction"]}

In [ ]:
train_test_overlap(train, test)

**Observation to discuss:** attacks are the *majority* class in both files, and
some test rows also appear (feature-for-feature) in training. We report this
rather than deleting rows, so results stay comparable with published work.

## 3. Attack categories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train["attack_cat"].value_counts().plot.barh(ax=axes[0], title="Training file")
test["attack_cat"].value_counts().plot.barh(ax=axes[1], title="Testing file")
plt.tight_layout()

## 4. Skewed numeric features (why we log-transform for LR/KNN/SVM)

In [ ]:
cols = ["dur", "sbytes", "dbytes", "rate", "sload", "smean"]
train[cols].describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.ravel(), cols):
    for lbl, name in config.CLASS_NAMES.items():
        ax.hist(np.log10(train.loc[train.label == lbl, c] + 1), bins=50, alpha=0.6, label=name)
    ax.set_title(f"log10({c}+1)")
axes[0, 0].legend()
plt.tight_layout()

## 5. A suspiciously strong feature: `sttl` (source time-to-live)

In [ ]:
pd.crosstab(train['sttl'], train['attack_cat']).loc[lambda d: d.sum(axis=1) > 500]

A few TTL values separate normal and attack traffic almost perfectly. This is
an artefact of how the UNSW testbed generated traffic, so a model that leans on
it may not generalise to other networks. Keep this in mind when reading the
feature-importance results.

## 6. Categorical features

In [ ]:
for c in config.CATEGORICAL_FEATURES:
    print(c, train[c].nunique(), "unique values; top:", train[c].value_counts().head(5).to_dict())